In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(f'../dataset/dataset-tickets-multi-lang-4-20k.csv', encoding='utf-8')

In [3]:
df = df[df['language'] == 'en'].copy()

In [4]:
X = df[['subject', 'body']]
y = df['priority']

In [39]:
X

,subject,body
1,Customer Support Inquiry,Seeking information on digital strategies that...
2,Data Analytics for Investment,I am contacting you to request information on ...
4,Security,"Dear Customer Support, I am reaching out to in..."
5,Concerns About Securing Medical Data on 2-in-1...,Inquiring about best practices for securing me...
7,Problem with Integration,"The integration stopped working unexpectedly, ..."
...,...,...
19992,Guidelines for Securing Medical Data in OBS St...,Seeking details on securing medical data using...
19993,NaN,Can you provide information on digital strateg...
19994,Support for Marketing Enhancements,Request for assistance in improving digital ma...
19995,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...


In [5]:
import re

urgent_words = {
    "urgent", "asap", "immediately", "critical",
    "терміново", "срочно", "dringend"
}

incident_words = {
    "crash", "breach", "loss", "blocked", "down",
    "failure", "outage", "error", "помилка"
}

negative_words = {
    "problem", "cannot", "failed", "failure",
    "unresolved", "not working", "issue"
}

action_phrases = {
    "please fix", "need help", "investigate",
    "restore", "resolve", "recover"
}

def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).lower()

def count_terms(text, terms):
    return sum(text.count(term) for term in terms)

def extract_priority_features(row):
    raw_subject = "" if pd.isna(row["subject"]) else str(row["subject"])
    raw_body = "" if pd.isna(row["body"]) else str(row["body"])
    subject = normalize_text(raw_subject)
    body = normalize_text(raw_body)
    combined_text = f"{subject} {body}"
    words = re.findall(r"\b\w+\b", combined_text)

    uppercase_words = re.findall(
        r"\b[A-ZА-ЯІЇЄ]{2,}\b",
        f"{raw_subject} {raw_body}"
    )

    subject_words = set(re.findall(r"\b\w+\b", subject))
    body_words = set(re.findall(r"\b\w+\b", body))
    union = subject_words | body_words

    return {
        "urgency_keyword_count": int(count_terms(combined_text, urgent_words)),
        "incident_keyword_count": int(count_terms(combined_text, incident_words)),
        "negative_word_count": int(count_terms(combined_text, negative_words)),
        "action_request_count": int(count_terms(combined_text, action_phrases)),
        "digit_count": int(
            sum(character.isdigit() for character in combined_text)
        ),
        "uppercase_word_ratio": round(
            float(len(uppercase_words) / len(words)) if words else 0.0,
            4
        ),
        "subject_body_similarity": round(
            float(len(subject_words & body_words) / len(union))
            if union else 0.0,
            4
        ),
    }

In [6]:
X['Subject + body'] = X['subject'] + ' ' + X['body']

In [7]:
X

,subject,body,Subject + body
1,Customer Support Inquiry,Seeking information on digital strategies that...,Customer Support Inquiry Seeking information o...
2,Data Analytics for Investment,I am contacting you to request information on ...,Data Analytics for Investment I am contacting ...
4,Security,"Dear Customer Support, I am reaching out to in...","Security Dear Customer Support, I am reaching ..."
5,Concerns About Securing Medical Data on 2-in-1...,Inquiring about best practices for securing me...,Concerns About Securing Medical Data on 2-in-1...
7,Problem with Integration,"The integration stopped working unexpectedly, ...",Problem with Integration The integration stopp...
...,...,...,...
19992,Guidelines for Securing Medical Data in OBS St...,Seeking details on securing medical data using...,Guidelines for Securing Medical Data in OBS St...
19993,NaN,Can you provide information on digital strateg...,NaN
19994,Support for Marketing Enhancements,Request for assistance in improving digital ma...,Support for Marketing Enhancements Request for...
19995,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,Assistance Needed for IFTTT Docker Integration...


In [8]:
feature_columns = [
    "urgency_keyword_count",
    "incident_keyword_count",
    "negative_word_count",
    "action_request_count",
    "digit_count",
    "uppercase_word_ratio",
    "subject_body_similarity",
]

X = X.drop(columns=[
    column for column in feature_columns if column in X.columns
])
priority_features = X.apply(
    extract_priority_features,
    axis=1,
    result_type="expand"
)
X = pd.concat([X, priority_features], axis=1)

In [44]:
X

,subject,body,Subject + body,urgency_keyword_count,incident_keyword_count,negative_word_count,action_request_count,digit_count,uppercase_word_ratio,subject_body_similarity
1,Customer Support Inquiry,Seeking information on digital strategies that...,Customer Support Inquiry Seeking information o...,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000
2,Data Analytics for Investment,I am contacting you to request information on ...,Data Analytics for Investment I am contacting ...,0.0,0.0,0.0,0.0,0.0,0.0273,0.0615
4,Security,"Dear Customer Support, I am reaching out to in...","Security Dear Customer Support, I am reaching ...",0.0,1.0,0.0,0.0,0.0,0.0088,0.0132
5,Concerns About Securing Medical Data on 2-in-1...,Inquiring about best practices for securing me...,Concerns About Securing Medical Data on 2-in-1...,0.0,0.0,0.0,0.0,10.0,0.0000,0.2791
7,Problem with Integration,"The integration stopped working unexpectedly, ...",Problem with Integration The integration stopp...,0.0,1.0,4.0,1.0,0.0,0.0213,0.0513
...,...,...,...,...,...,...,...,...,...,...
19992,Guidelines for Securing Medical Data in OBS St...,Seeking details on securing medical data using...,Guidelines for Securing Medical Data in OBS St...,0.0,0.0,0.0,0.0,2.0,0.0667,0.2083
19993,NaN,Can you provide information on digital strateg...,NaN,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000
19994,Support for Marketing Enhancements,Request for assistance in improving digital ma...,Support for Marketing Enhancements Request for...,0.0,0.0,0.0,0.0,0.0,0.0000,0.0577
19995,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,Assistance Needed for IFTTT Docker Integration...,0.0,0.0,5.0,2.0,0.0,0.0385,0.0714


In [45]:
for column in feature_columns:
    print(f"{column} - Max = {X[column].max()}, Min = {X[column].min()}, Mean = {X[column].mean()}, Median = {X[column].median()}, Std = {X[column].std()}")

urgency_keyword_count - Max = 4.0, Min = 0.0, Mean = 0.09427157594565126, Median = 0.0, Std = 0.3549490429198401
incident_keyword_count - Max = 8.0, Min = 0.0, Mean = 0.4343705443260924, Median = 0.0, Std = 0.8352550372571701
negative_word_count - Max = 10.0, Min = 0.0, Mean = 1.381615365260421, Median = 1.0, Std = 1.750491820746293
action_request_count - Max = 4.0, Min = 0.0, Mean = 0.30395034806676174, Median = 0.0, Std = 0.6000394173294844
digit_count - Max = 22.0, Min = 0.0, Mean = 0.37423467248175796, Median = 0.0, Std = 1.4031432615846158
uppercase_word_ratio - Max = 0.375, Min = 0.0, Mean = 0.005648704185188291, Median = 0.0, Std = 0.018231975234262645
subject_body_similarity - Max = 1.0, Min = 0.0, Mean = 0.08758751991948335, Median = 0.069, Std = 0.08154320414461964


In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X[feature_columns], y, test_size=0.2, random_state=42, stratify=y)

In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

In [12]:
baseline_params = {
    "criterion": "gini",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
}

model = DecisionTreeClassifier(
    random_state=42,
    **baseline_params
)
model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)
cross_val_scores = cross_val_score(model, X_train, y_train, cv=5)

print("Baseline hyperparameters:", baseline_params)
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Cross-validation accuracy: {cross_val_scores.mean():.4f} +/- {cross_val_scores.std():.4f}")

Baseline hyperparameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Train accuracy: 0.7683
Test accuracy: 0.3874
Cross-validation accuracy: 0.3919 +/- 0.0039


In [49]:
cross_val_scores

array([0.39046122, 0.38836478, 0.39465409, 0.38804405, 0.39800734])

In [13]:
depth_results = []

depth_values = [1, 2, 3, 4, 5, 7, 10, 15, None]

for max_depth in depth_values:
    depth_model = DecisionTreeClassifier(
        criterion="gini",
        max_depth=max_depth,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
    )
    depth_cv_scores = cross_val_score(
        depth_model,
        X_train,
        y_train,
        cv=5,
    )

    depth_results.append({
        "criterion": "gini",
        "max_depth": max_depth,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "cv_accuracy_mean": round(depth_cv_scores.mean(), 4),
        "cv_accuracy_std": round(depth_cv_scores.std(), 4),
    })

depth_results = pd.DataFrame(depth_results).sort_values(
    by="cv_accuracy_mean",
    ascending=False,
).reset_index(drop=True)

best_depth_row = depth_results.iloc[0]
best_max_depth = best_depth_row["max_depth"]
if pd.isna(best_max_depth):
    best_max_depth = None
else:
    best_max_depth = int(best_max_depth)

best_depth_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=best_max_depth,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
)
best_depth_model.fit(X_train, y_train)

best_train_accuracy = accuracy_score(
    y_train,
    best_depth_model.predict(X_train),
)
best_test_accuracy = accuracy_score(
    y_test,
    best_depth_model.predict(X_test),
)

print("Best max_depth selected by cross-validation:", best_max_depth)
print(f"Best CV accuracy: {best_depth_row['cv_accuracy_mean']:.4f}")
print(f"Selected model train accuracy: {best_train_accuracy:.4f}")
print(f"Selected model test accuracy: {best_test_accuracy:.4f}")

depth_results

Best max_depth selected by cross-validation: 5
Best CV accuracy: 0.4611
Selected model train accuracy: 0.4698
Selected model test accuracy: 0.4516


,criterion,max_depth,min_samples_split,min_samples_leaf,cv_accuracy_mean,cv_accuracy_std
0,gini,5.0,2,1,0.4611,0.0084
1,gini,3.0,2,1,0.4605,0.0095
2,gini,2.0,2,1,0.4595,0.0093
3,gini,1.0,2,1,0.4582,0.0082
4,gini,4.0,2,1,0.4579,0.0107
5,gini,7.0,2,1,0.4547,0.0097
6,gini,10.0,2,1,0.4526,0.0072
7,gini,15.0,2,1,0.4409,0.0094
8,gini,NaN,2,1,0.3919,0.0039


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_train_predictions = rf_model.predict(X_train)
rf_test_predictions = rf_model.predict(X_test)

rf_train_accuracy = accuracy_score(y_train, rf_train_predictions)
rf_test_accuracy = accuracy_score(y_test, rf_test_predictions)
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)

print("Random Forest Baseline:")
print(f"Train accuracy: {rf_train_accuracy:.4f}")
print(f"Test accuracy: {rf_test_accuracy:.4f}")
print(f"Cross-validation accuracy: {rf_cv_scores.mean():.4f} +/- {rf_cv_scores.std():.4f}")

Random Forest Baseline:
Train accuracy: 0.7683
Test accuracy: 0.4046
Cross-validation accuracy: 0.4052 +/- 0.0047


In [15]:
rf_feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_,
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

rf_feature_importance

,feature,importance
0,subject_body_similarity,0.635296
1,uppercase_word_ratio,0.137204
2,negative_word_count,0.091708
3,incident_keyword_count,0.045918
4,digit_count,0.042290
5,action_request_count,0.029611
6,urgency_keyword_count,0.017973


In [16]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
)
gb_model.fit(X_train, y_train)

gb_train_predictions = gb_model.predict(X_train)
gb_test_predictions = gb_model.predict(X_test)

gb_train_accuracy = accuracy_score(y_train, gb_train_predictions)
gb_test_accuracy = accuracy_score(y_test, gb_test_predictions)
gb_cv_scores = cross_val_score(gb_model, X_train, y_train, cv=5)

print("Gradient Boosting Baseline:")
print(f"Train accuracy: {gb_train_accuracy:.4f}")
print(f"Test accuracy: {gb_test_accuracy:.4f}")
print(f"Cross-validation accuracy: {gb_cv_scores.mean():.4f} +/- {gb_cv_scores.std():.4f}")

Gradient Boosting Baseline:
Train accuracy: 0.4878
Test accuracy: 0.4516
Cross-validation accuracy: 0.4575 +/- 0.0115


In [17]:
comparison_results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Gradient Boosting"],
    "Train Accuracy": [
        best_train_accuracy,
        rf_train_accuracy,
        gb_train_accuracy,
    ],
    "Test Accuracy": [
        best_test_accuracy,
        rf_test_accuracy,
        gb_test_accuracy,
    ],
    "CV Accuracy Mean": [
        best_depth_row["cv_accuracy_mean"],
        rf_cv_scores.mean(),
        gb_cv_scores.mean(),
    ],
    "CV Accuracy Std": [
        best_depth_row["cv_accuracy_std"],
        rf_cv_scores.std(),
        gb_cv_scores.std(),
    ],
})

comparison_results.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)

,Model,Train Accuracy,Test Accuracy,CV Accuracy Mean,CV Accuracy Std
0,Decision Tree,0.469805,0.451572,0.461100,0.008400
1,Gradient Boosting,0.487838,0.451572,0.457540,0.011513
2,Random Forest,0.768295,0.404612,0.405222,0.004696


In [18]:
param_grid_rf = {
    "max_depth": [5, 10, 15, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "n_estimators": [50, 100, 200],
}

rf_search_results = []

for max_depth in param_grid_rf["max_depth"]:
    for min_samples_leaf in param_grid_rf["min_samples_leaf"]:
        for n_estimators in param_grid_rf["n_estimators"]:
            rf_candidate = RandomForestClassifier(
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                n_estimators=n_estimators,
                random_state=42,
                n_jobs=-1,
            )
            
            cv_scores = cross_val_score(
                rf_candidate,
                X_train,
                y_train,
                cv=5,
            )
            
            rf_search_results.append({
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "n_estimators": n_estimators,
                "cv_accuracy_mean": round(cv_scores.mean(), 4),
                "cv_accuracy_std": round(cv_scores.std(), 4),
            })

rf_search_df = pd.DataFrame(rf_search_results)
best_rf_params = rf_search_df.loc[rf_search_df["cv_accuracy_mean"].idxmax()]

print("Best RF hyperparameters (by CV accuracy on X_train):")
print(best_rf_params)

best_rf_model = RandomForestClassifier(
    max_depth=int(best_rf_params["max_depth"]) if best_rf_params["max_depth"] is not None else None,
    min_samples_leaf=int(best_rf_params["min_samples_leaf"]),
    n_estimators=int(best_rf_params["n_estimators"]),
    random_state=42,
    n_jobs=-1,
)
best_rf_model.fit(X_train, y_train)

best_rf_train_accuracy = accuracy_score(y_train, best_rf_model.predict(X_train))
best_rf_test_accuracy = accuracy_score(y_test, best_rf_model.predict(X_test))

print(f"\nBest RF model on X_test:")
print(f"Train accuracy: {best_rf_train_accuracy:.4f}")
print(f"Test accuracy: {best_rf_test_accuracy:.4f}")

print(f"\nTotal configurations tested: {len(rf_search_df)}")

rf_search_df

Best RF hyperparameters (by CV accuracy on X_train):
max_depth             5.0000
min_samples_leaf      8.0000
n_estimators        200.0000
cv_accuracy_mean      0.4638
cv_accuracy_std       0.0105
Name: 11, dtype: float64

Best RF model on X_test:
Train accuracy: 0.4706
Test accuracy: 0.4474

Total configurations tested: 48


,max_depth,min_samples_leaf,n_estimators,cv_accuracy_mean,cv_accuracy_std
0,5.0,1,50,0.4632,0.0091
1,5.0,1,100,0.4637,0.0109
2,5.0,1,200,0.4624,0.0099
3,5.0,2,50,0.4629,0.0096
4,5.0,2,100,0.4627,0.0112
5,5.0,2,200,0.4624,0.0095
6,5.0,4,50,0.4620,0.0094
7,5.0,4,100,0.4636,0.0101
8,5.0,4,200,0.4629,0.0100
9,5.0,8,50,0.4637,0.0099
